In [1]:
import pandas as pd

from main import calculate_pnl

usdt_to_gbp_df = pd.read_csv('./data/usd_gbp.csv', index_col=0)
# df[Price   ,     Date,  USD_to_GBP]
usdt_to_gbp_df = usdt_to_gbp_df.iloc[1:]
usdt_to_gbp_df['Date'] = pd.to_datetime(usdt_to_gbp_df['Date'])
usdt_to_gbp_df.set_index('Date')
usdt_to_gbp_df = usdt_to_gbp_df.set_index('Date')
usdt_to_gbp_df = usdt_to_gbp_df.sort_index()

bnb_to_usdt_df = pd.read_csv('./data/bnb_usdt.csv', index_col=0)
# df[datetime  close]
bnb_to_usdt_df['datetime'] = pd.to_datetime(bnb_to_usdt_df['datetime'])
bnb_to_usdt_df = bnb_to_usdt_df.set_index('datetime')
bnb_to_usdt_df = bnb_to_usdt_df.sort_index()

trades_spot_df = calculate_pnl('spot', usdt_to_gbp_df, bnb_to_usdt_df)
trades_margin_df = calculate_pnl('margin', usdt_to_gbp_df, bnb_to_usdt_df)
# open_time,close_time,symbol,qty,profit,commission_usdt,commission_bnb
df_combined = pd.concat([trades_spot_df, trades_margin_df], ignore_index=True)

interest_df = pd.read_csv('./data/raw/interest/interest_margin.csv')
interest_df['interestAccuredTime'] = pd.to_datetime(interest_df['interestAccuredTime'])
interest_df = pd.merge_asof(
    interest_df.sort_values('interestAccuredTime'),
    bnb_to_usdt_df[['close']].sort_index().rename(columns={'close': 'bnb_usdt'}),
    left_on='interestAccuredTime',
    right_index=True,
    direction='backward'
)
interest_df = pd.merge_asof(
    interest_df.sort_values('interestAccuredTime'),
    usdt_to_gbp_df[['USD_to_GBP']].sort_index().rename(columns={'USD_to_GBP': 'usd_to_gbp'}),
    left_on='interestAccuredTime',
    right_index=True,
    direction='backward'
)
interest_df['interest_in_usd'] = interest_df['interest'] * interest_df['bnb_usdt']
interest_df['interest_in_gbp'] = interest_df['interest_in_usd'] * interest_df['usd_to_gbp']
df_combined['disposal_date'] = pd.to_datetime(df_combined['disposal_date'])
trades_sorted = df_combined[['disposal_date']].reset_index().rename(columns={'index': 'trade_id'}).sort_values(
    'disposal_date')
interest_sorted = interest_df.sort_values('interestAccuredTime')
merged = pd.merge_asof(
    interest_sorted,
    trades_sorted,
    left_on='interestAccuredTime',
    right_on='disposal_date',
    direction='nearest'
)
assigned = merged.groupby('trade_id')['interest_in_gbp'].sum().reset_index()
df_combined = df_combined.reset_index().rename(columns={'index': 'trade_id'})
df_combined = df_combined.merge(assigned, on='trade_id', how='left').fillna({'interest_in_gbp': 0})
df_combined['cost_in_gbp'] += df_combined['interest_in_gbp']
df_combined['net_profit_in_gbp'] -= df_combined['interest_in_gbp']
df_combined = df_combined.drop(columns=['trade_id'])

print(trades_margin_df.tail())

c:\Myfiles\MyProjects\uk-crypto-tax-report\src\utils.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ticker = yf.download("GBPUSD=X", start=start, end=end, interval="1d")
[*********************100%***********************]  1 of 1 completed


ACAUSDT_margin_trades.csv:
{'datetime': Timestamp('2024-12-16 01:43:17.623000'), 'symbol': 'ACAUSDT', 'qty': 181.95999999999998, 'profit': -0.00909800000000026, 'commission_usdt': 0, 'commission_bnb': 2.2835114449934455e-05, 'total_proceeds': 10.999482, 'total_cost': 11.008579999999998, 'total_gain_loss': -0.00909799999999894, 'asset': 'ACA', 'disposal_date': Timestamp('2024-12-16 01:43:17.623000')}

ADAUSDT_margin_trades.csv:
{'datetime': Timestamp('2024-11-08 20:43:18.556000'), 'symbol': 'ADAUSDT', 'qty': 50.0, 'profit': -0.0025000000000011124, 'commission_usdt': 0, 'commission_bnb': 2.7686153846153846e-05, 'total_proceeds': 10.9675, 'total_cost': 10.97, 'total_gain_loss': -0.002500000000001279, 'asset': 'ADA', 'disposal_date': Timestamp('2024-11-08 20:43:18.556000')}

API3USDT_margin_trades.csv:
{'datetime': Timestamp('2024-05-11 18:43:25.178000'), 'symbol': 'API3USDT', 'qty': 9.18, 'profit': -0.002389999999999736, 'commission_usdt': 0, 'commission_bnb': 2.780316666666667e-05, 'tota

In [28]:
a = set(assigned['trade_id'].unique())

print(len(a))

2049


In [30]:
interest_df.head()

,Unnamed: 0,txId,interestAccuredTime,asset,rawAsset,principal,interest,interestRate,type,bnb_usdt,usd_to_gbp,interest_in_usd,interest_in_gbp
988,988,1697563309527143845,2024-04-05 23:00:00,BNB,BTTC,0.30,1.000000e-08,0.000505,PERIODIC_CONVERTED,580.0,0.791127,0.000006,0.000005
987,987,1697594276241741221,2024-04-06 00:00:00,BNB,BTTC,0.30,1.000000e-08,0.000485,PERIODIC_CONVERTED,577.5,0.791127,0.000006,0.000005
985,985,1697627124149564461,2024-04-06 01:00:00,BNB,VIC,968.09,9.805000e-05,0.001390,ON_BORROW_CONVERTED,577.6,0.791127,0.056634,0.044804
986,986,1697625182824371621,2024-04-06 01:00:00,BNB,BTTC,0.30,1.000000e-08,0.000556,PERIODIC_CONVERTED,577.6,0.791127,0.000006,0.000005
983,983,1697656553266222501,2024-04-06 02:00:00,BNB,VIC,968.09,1.253300e-04,0.001845,PERIODIC_CONVERTED,577.4,0.791127,0.072366,0.057250


In [ ]:
df_combined = df_combined.reset_index().rename(columns={'index': 'trade_id'})
df_combined = df_combined.merge(assigned, on='trade_id', how='left').fillna({'interest_in_gbp': 0})
df_combined['cost_in_gbp'] += df_combined['interest_in_gbp']
df_combined['net_profit_in_gbp'].sum()
 
